# Deploy the demo to a free CPU Hugging Face Space

Self-contained: creates a Gradio Space, uploads the app + data, and points it at
your merged model on the Hub. Run top to bottom. Needs an **HF WRITE token**
(huggingface.co/settings/tokens).

Prereq: the MERGED model must already be on the Hub (step 11 of run_all_pipeline,
or run the optional push cell below).

## 1. Install + log in

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # paste your hf_... WRITE token into the hidden box

## 2. Config — set your names

In [ ]:
HF_USER  = "gabriel-xiong"
MODEL_ID = f"{HF_USER}/apbio-item-generator-qwen3-1.7b"   # your MERGED model repo
SPACE_ID = f"{HF_USER}/apbio-item-generator"              # the Space to create

## 3. (Optional) push the merged model to the Hub
Skip if you already did step 11 of the pipeline. Requires the model in memory
(`model`, `tok`) or reload the adapter first.

In [ ]:
# from unsloth import FastLanguageModel
# model, tok = FastLanguageModel.from_pretrained('outputs/gen_lora', max_seq_length=1536, load_in_4bit=True)
# model.push_to_hub_merged(MODEL_ID, tok, save_method='merged_16bit')

## 4. Get the app files from GitHub

In [ ]:
import os
if not os.path.isdir("QuestionGen"):
    !git clone -q https://github.com/gabriel-xiong/QuestionGen.git
# point app.py at your model
src = open('QuestionGen/scripts/app.py', encoding='utf-8').read()
import re
src = re.sub(r'MODEL_ID = "[^"]*"', f'MODEL_ID = "{MODEL_ID}"', src, count=1)
open('app.py','w',encoding='utf-8').write(src)
open('requirements.txt','w').write('transformers
torch
gradio
')
os.makedirs('data', exist_ok=True)
import shutil; shutil.copy('QuestionGen/data/apbio_misconceptions.json','data/apbio_misconceptions.json')
print('prepared: app.py, requirements.txt, data/apbio_misconceptions.json')

## 5. Create the Space (free CPU) and upload the files

In [ ]:
from huggingface_hub import HfApi, create_repo
create_repo(SPACE_ID, repo_type='space', space_sdk='gradio', exist_ok=True)
api = HfApi()
for f in ['app.py','requirements.txt','data/apbio_misconceptions.json']:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=SPACE_ID, repo_type='space')
    print('uploaded', f)
print('
Space building at: https://huggingface.co/spaces/' + SPACE_ID)
print('First build takes a few minutes; the URL is permanent and free.')

## Done
Your Space is live at `https://huggingface.co/spaces/<user>/apbio-item-generator`.
Free CPU: ~10-30s per generation, sleeps after ~48h idle and wakes on visit.
For faster inference, switch the Space hardware to ZeroGPU/GPU in its Settings —
no code change needed.